In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.dates as mdates
# import constants for the days of the week
from matplotlib.dates import MO, TU, WE, TH, FR, SA, SU
import matplotlib.colors as mcolors
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec

# RCEW Data

## Importing precipipitation data

In [ ]:
def import_and_clean_reynolds_data(path, header):
    df = pd.read_csv(path, header=header, encoding='latin1', na_values=[-999, 'Z'])

    # Remove rows with invalid datetime values
    df = df[df['datetime'].str.match(r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}$', na=False)]

    # Convert the datetime column to a datetime object
    df['datetime'] = pd.to_datetime(df['datetime'])

    # Set the datetime column as the index
    return df.set_index('datetime')

In [ ]:
rmsp3b_precip = import_and_clean_reynolds_data('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Precipitation/reynolds-creek-rmsp3b-hourly-precipitation.dat', 20)
precip_176 = import_and_clean_reynolds_data('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Precipitation/reynolds-creek-176-hourly-precipitation.dat', 20)
precip_125 = import_and_clean_reynolds_data('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Precipitation/reynolds-creek-125-hourly-precipitation.dat', 20)

rmsp3b_precip

# Importing Soil Temp

In [ ]:
soil_176 = import_and_clean_reynolds_data('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Soil/reynolds-creek-176-hourly-soil-temperature.dat', 27)


In [ ]:
fig, ax = plt.subplots(figsize=(11,5))

soil_176['2024-12-01':'2025-05-15'].plot(y=['stm005', 'stm010', 'stm020', 'stm030', 'stm040', 'stm050', 'stm060', 'stm090', 'stm120', 'stm180'], ax=ax)
ax.axhline(0, label= 'Freezing', linestyle='--')
ax.set_ylim(-1, 1)

## Importing soil mosture

In [ ]:
soilmoisture_176 = import_and_clean_reynolds_data('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Soil/reynolds-creek-176-provisional-hydraprobe.dat', 27)
soilmoisture_mbsec = import_and_clean_reynolds_data('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Soil/reynolds-creek-mbsec-provisional-hydraprobe.dat', 27)
soilmoisture_mbsec_hourly = soilmoisture_mbsec.resample('1h').mean()
soilmoisture_176_hourly = soilmoisture_176.resample('1h').mean()


In [ ]:
fig, ax= plt.subplots(nrows=2, figsize=(11,8.5))

soilmoisture_mbsec.loc['02/2/2025':'08/01/2025'].plot(y=['wat005','wat015', 'wat030', 'wat060', 'wat090'], ax=ax[0], ylabel = 'Volumetric Water Content', title='mbsec Soil Moisture')
soilmoisture_mbsec.loc['02/20/2025':'08/01/2025'].plot(y=['temp005','temp015', 'temp030', 'temp060', 'temp090'], ax=ax[1], ylabel = 'Temperature (C)', title='mbsec Soil Temperature')

ax[1].set_ylim(-.1, 1)
fig.tight_layout()

In [ ]:
fig, ax= plt.subplots(nrows=2, figsize=(11,8.5))

soilmoisture_176.loc['02/23/2025':'05/12/2025'].plot(y=['wat005'], ax=ax[0], ylabel = 'Volumetric Water Content', title='176 Soil Moisture')
soilmoisture_176.loc['02/23/2025':'05/12/2025'].plot(y=['temp005'], ax=ax[1], ylabel = 'Temperature (C)', title='176 Soil Temperature')

ax[1].set_ylim(-.1, 1)
fig.tight_layout()

## Importing snow depth

In [ ]:
snow_176 = import_and_clean_reynolds_data('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Snow/reynolds-creek-176-snow-depth.dat', 18)
rmsp3b_snow = import_and_clean_reynolds_data('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Snow/reynolds-creek-rmsp3b-snow-depth.dat', 18)

snow_176


# Importing SWE and merging

In [ ]:
rmsp_swe= pd.read_excel('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Snow/RMSP_SWE.xlsx', index_col='Datetime', parse_dates=True)
rmsp_swe['SWE_corr mm'] = rmsp_swe['SWE_corr']*25.4




## Importing Weather Data

In [ ]:
rmsp3b_weather = import_and_clean_reynolds_data('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Climate/reynolds-creek-rmsp3b-climate-l1.dat', 24)
weather_176 = import_and_clean_reynolds_data('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Climate/reynolds-creek-176-climate-l1.dat', 39)
weather_125 = import_and_clean_reynolds_data('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Climate/reynolds-creek-125-climate-l1.dat', 29)

rmsp3b_weather

## Joining Precip and Weather, Computing Wet Bulb

In [ ]:
rmsp3_combined = rmsp3b_precip.join(rmsp3b_weather.resample('1h', label='right').mean(), how='outer', sort=True)
rmsp3_combined = rmsp3_combined.join(rmsp3b_snow.resample('1h', label='right').mean(), how='outer', sort=True)
rmsp3_combined = rmsp3_combined.join(rmsp_swe.resample('1h', label='right').mean(), how='outer', sort=True)
rmsp3_combined = rmsp3_combined.fillna({'SWE_corr': 0, 'SWE_corr mm':0})


combined_176 = precip_176.join(weather_176.resample('1h', label='right').mean(), how='outer', sort=True)
combined_176 = combined_176.join(snow_176.resample('1h', label='right').mean(), how='outer', sort=True)
combined_176 = combined_176.join(soil_176, how='outer', sort=True)
combined_176 = combined_176.join(soilmoisture_176_hourly, how='outer', sort=True)

combined_125 = precip_125.join(weather_125.resample('1h', label='right').mean(), how='outer', sort=True)


rmsp3_combined['2025-02-01':'2025-02-02']
combined_125

### Calculating Dewpoint and Precipitation Phase

In [ ]:
def calculate_dewpoint(df, elevation=3, b=17.625, c=243.04):
    gamma = np.log(df['hum'+ str(elevation)]/100)+b*df['tmp'+str(elevation)]/(c+df['tmp'+str(elevation)])
    df['dpt'+str(elevation)] = c*gamma/(b-gamma)
    return df

def calculate_precipitation_phase(df, elevation=3,dewpoint_threshold=0):
    dewpoint_column = 'dpt' + str(elevation)
    df['ppta_snow'] = 0
    df['ppta_rain'] = 0
    snowrows =  df[dewpoint_column] <= dewpoint_threshold
    df['ppta_snow'].loc[snowrows] = df['ppta']
    df['ppta_rain'].loc[~snowrows] = df['ppta']
    return df

def calculate_cum_swe(df):
    def get_water_year(dt):
        # dt can be a pandas Timestamp or datetime
        return dt.year + 1 if dt.month >= 10 else dt.year
    
    df['water_year'] = df.index.map(get_water_year)
    df['cum_swe'] = df.groupby('water_year').transform(lambda x: x.cumsum())['ppta_snow']

    return df


- calculating dewpoint with Magnus formula
- calculating precipitation phase according to Nayak et al, 2008

In [ ]:
rmsp3_combined = calculate_dewpoint(rmsp3_combined)
rmsp3_combined = calculate_precipitation_phase(rmsp3_combined)
rmsp3_combined = calculate_cum_swe(rmsp3_combined)

combined_176 = calculate_dewpoint(combined_176)
combined_176 = calculate_precipitation_phase(combined_176)
combined_176 = calculate_cum_swe(combined_176)
combined_176

combined_125 = calculate_dewpoint(combined_125)
combined_125 = calculate_precipitation_phase(combined_125)


In [ ]:
fig, ax= plt.subplots(figsize = (11,8.5))
#rmsp3_combined.loc['2025-01-23'].plot(y=['ppta_rain'],kind='bar', ax=ax, rot=45)

ax.bar(rmsp3_combined.loc['2025-02-21':].index,rmsp3_combined.loc['2025-02-21':]['ppta_rain'], width = pd.Timedelta(1,'hour'),color='blue')
ax.bar(rmsp3_combined.loc['2025-02-21':].index,rmsp3_combined.loc['2025-02-21':]['ppta_snow'],width = pd.Timedelta(1,'hour'), color='orange')

# Set x-ticks to 1-week intervals and format them
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=MO, interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))

plt.xticks(rotation=45)
#plt.tight_layout()
#plt.show()


# NADP Data

## Import Data

In [ ]:
nadp_weekly = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/NADP Data/NTN-id11-w-s-mg.csv', parse_dates=True, na_values=-9, index_col = 'dateOn')
nadp_weekly

In [ ]:
fig, ax = plt.subplots(nrows = 2, figsize=(11,8.5), sharex=True)
nadp_weekly.loc['11/1/2024':].reset_index().plot(x='dateOff', y='NO3', kind='bar', ylim=(0,1.2),ax=ax[0])
nadp_weekly.loc['11/1/2024':].reset_index().plot(x='dateOff', y='ppt', kind='bar',ax=ax[1])

fig.tight_layout()

In [ ]:
combined_176

## Extract Valid Weeks

In [ ]:
# filter out invalid invalcodes
nadp_weekly_valid = nadp_weekly.loc[~(nadp_weekly['invalcode'].str.contains('b|u|f|c|v|e|l|i|n|p|x', regex=True))]

# keep only valid valcodes
nadp_weekly_valid = nadp_weekly_valid.loc[nadp_weekly_valid['valcode'].str.contains('w')]

nadp_weekly_valid['valcode'].unique()

In [ ]:
nadp_weekly_valid['invalcode'].unique()

In [ ]:
fig, ax = plt.subplots(nrows = 2, figsize=(11,8.5), sharex=True)
nadp_weekly_valid.loc['11/1/2024':].reset_index().plot(x='dateOff', y='NO3', kind='bar', ylim=(0,1.2),ax=ax[0])
nadp_weekly_valid.loc['11/1/2024':].reset_index().plot(x='dateOff', y='ppt', kind='bar',ax=ax[1])

fig.tight_layout()

In [ ]:
nadp_weekly.columns

In [ ]:
jan = nadp_weekly_valid['2025-01-01':'2025-01-31']

In [ ]:
numeric_cols = ['ph', 'Conduc', 'Ca', 'Mg', 'K', 'Na', 'NH4', 'NO3', 'Cl', 'SO4', 'Br', 'svol', 'ppt', 'subppt']

In [ ]:
def calc_weighted_mean(df, weight):
    if df[weight].sum() <= 0:
        return pd.Series([np.nan] * len(df.columns), index=df.columns)

    else:
        result = np.average(df, axis=0, weights=df[weight])
        return pd.Series(result, index = df.columns)


nadp_monthly= nadp_weekly_valid[numeric_cols].groupby(pd.Grouper(freq='ME')).apply(lambda x: calc_weighted_mean(x, 'subppt'))
#monthly_grouped = pd.DataFrame(data = monthly_grouped_array[1],index=monthly_grouped_array[0], columns = numeric_cols)
nadp_monthly

## Merge with Precip and Calculate Concentration

In [ ]:
nadp_hourly = nadp_monthly.resample('1h').bfill()
nadp_hourly

In [ ]:
rmsp3_combined = pd.merge(rmsp3_combined, nadp_hourly, right_index=True, left_index=True, how='outer')
combined_176 = pd.merge(combined_176, nadp_hourly, right_index=True, left_index=True, how='outer')
combined_125 = pd.merge(combined_125, nadp_hourly, right_index=True, left_index=True, how='outer')


In [ ]:
def calculate_deposition(df, precip_type, precip_col, analytes):
    for analyte in analytes:
        dep_col = precip_type + ' ' + analyte + ' Deposition'
        df[dep_col]  = df[precip_col] * df[analyte]/100
        cum_dep_col = 'Cumulative ' + precip_type + ' ' + analyte + ' Deposition'
        df[cum_dep_col] = df.groupby('water_year').transform(lambda x: x.cumsum())[dep_col]
        df['Snowpack ' + analyte+ ' Concentration'] = df[cum_dep_col]/df['cum_swe']*100
    return df

In [ ]:
rmsp3_combined = calculate_deposition(rmsp3_combined, 'Snow', 'ppta_snow', ['NO3', 'NH4'])
combined_176 = calculate_deposition(combined_176, 'Snow', 'ppta_snow', ['NO3', 'NH4'])
rmsp3_combined['Cumulative Snow TIN Deposition']  = rmsp3_combined['Cumulative Snow NO3 Deposition'] + rmsp3_combined['Cumulative Snow NH4 Deposition']
combined_176['Cumulative Snow TIN Deposition']  = combined_176['Cumulative Snow NO3 Deposition'] + combined_176['Cumulative Snow NH4 Deposition']

In [ ]:
rme_snow = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/Nutrient Analyzer Data/Processed Data/rme_snow_aa500.csv',index_col = 'Sample Datetime',parse_dates=True)
rme_snow

In [ ]:
rme_avg = rme_snow['Nitrate mean'].describe()
rme_avg = pd.DataFrame([rme_avg], index = [pd.to_datetime('2025-04-02')])
rme_avg.index.name = 'Sample Datetime'
rme_avg['err'] = 2* rme_avg['std']
rme_avg


In [ ]:
rme_avg_nh4 = rme_snow['Ammonium mean'].describe()
rme_avg_nh4 = pd.DataFrame([rme_avg_nh4], index = [pd.to_datetime('2025-04-02')])
rme_avg_nh4.index.name = 'Sample Datetime'
rme_avg_nh4['err'] = 2* rme_avg_nh4['std']
rme_avg_nh4

In [ ]:
fig, ax = plt.subplots(nrows=4, figsize = (11,8.5))

start_date = '10/01/2024'
end_date = '05/12/2025'

second_y = ax[0].twinx()
third_y = ax[0].twinx()

combined_176.loc[start_date:end_date].plot(y='sno', ax=ax[0], x_compat=True, label='Snow Depth', ylabel = 'Snow Depth (cm)', title= '2025 WY Snowpack Development at RME Ridgetop', color='lightsteelblue')
combined_176.loc[start_date:end_date].plot(y='cum_swe', ax=third_y, x_compat=True, label='Cumulative Snowfall WE (mm)', ylabel = 'Cumulative Snowfall WE (mm)', legend=None, color='cornflowerblue')

second_y.bar(combined_176.loc[start_date:end_date].index,combined_176.loc[start_date:end_date]['ppta_rain'], width = pd.Timedelta(1,'hour'),color='royalblue', label='Rain')
second_y.bar(combined_176.loc[start_date:end_date].index,combined_176.loc[start_date:end_date]['ppta_snow'], width = pd.Timedelta(1,'hour'),color='tab:orange', label='Snow')
second_y.set_ylim(0,10)
ax[0].set_ylim(-5, 250)
second_y.set_ylabel('Hourly Precipitation (mm)')
second_y.invert_yaxis()
third_y.spines['right'].set_position(('axes', 1.1)) # Adjust the '1.15' value as needed

# Combine legends from both axes
handles1, labels1 = ax[0].get_legend_handles_labels()  # Primary y-axis
handles2, labels2 = second_y.get_legend_handles_labels()  # Secondary y-axis
handles3, labels3 = third_y.get_legend_handles_labels()
handles = handles1 + handles2 + handles3
labels = labels1 + labels2+ labels3

# Add the combined legend
ax[0].legend(handles, labels, loc='upper left')

#rmsp3_combined.loc[start_date:end_date].plot(y='sno', ax=ax[1], x_compat=True, label='Snow Depth', ylabel = 'Snow Depth (cm)', title='RMSP3 Snowpack Development')
#second_y2.bar(rmsp3_combined.loc[start_date:end_date].index,rmsp3_combined.loc[start_date:end_date]['ppta_rain'], width = pd.Timedelta(1,'hour'),color='royalblue', label='Rain')
#second_y2.bar(rmsp3_combined.loc[start_date:end_date].index,rmsp3_combined.loc[start_date:end_date]['ppta_snow'], width = pd.Timedelta(1,'hour'),color='tab:orange', label='Snow')
#second_y2.set_ylim(0,10)
#ax[1].set_ylim(-5, 250)
#second_y2.set_ylabel('Hourly Precipitation (mm)')
#second_y2.invert_yaxis()

combined_176.loc[start_date:end_date].plot(y='Cumulative Snow NO3 Deposition', title='2025 WY Snow NO3 Deposition at RME Ridgetop', ax=ax[3], ylabel='Snow N Deposition (kg-N/ha)', x_compat=True)
combined_176.loc[start_date:end_date].plot(y='Cumulative Snow NH4 Deposition', title='2025 WY Snow NO3 Deposition at RME Ridgetop', ax=ax[3], ylabel='Snow N Deposition (kg-N/ha)', x_compat=True)
combined_176.loc[start_date:end_date].plot(y='Cumulative Snow TIN Deposition', title='2025 WY Snow NO3 Deposition at RME Ridgetop', ax=ax[3], ylabel='Snow N Deposition (kg-N/ha)', x_compat=True)

combined_176.loc[start_date:end_date].plot(y='NO3', ax=ax[1], ylabel='Monthly Average Precipitation Nitrate Concentration',  label = 'Precipitation NO3', x_compat=True)
combined_176.loc[start_date:end_date].plot(y='NH4', ax=ax[1], ylabel='Monthly Average Precipitation Nitrate Concentration',  label = 'Precipitation NH4', x_compat=True)

combined_176.loc[start_date:end_date].plot(y='Snowpack NO3 Concentration', title='2025 WY Snowpack Nitrate at RME Ridgetop', label = 'Average Snowpack NO3', ax=ax[2], ylabel=' NO3 Concentration (mg/L)',  x_compat=True)
combined_176.loc[start_date:end_date].plot(y='Snowpack NH4 Concentration', title='2025 WY Snowpack Nitrate at RME Ridgetop', label = 'Average Snowpack NH4', ax=ax[2], ylabel=' NO3 Concentration (mg/L)',  x_compat=True)

rme_avg.reset_index().plot(y='mean', x='Sample Datetime', kind='scatter', yerr='err',ax=ax[2], label = 'Measured Snowpack NO3', color = 'tab:green')
rme_avg_nh4.reset_index().plot(y='mean', x='Sample Datetime', kind='scatter', yerr='err',ax=ax[2], label = 'Measured Snowpack NH4', color='blue')


fig.tight_layout()

In [ ]:
combined_176['2025-05-05':'2025-05-06']

# Export

In [ ]:
rmsp3_combined.index.name = 'Datetime'
combined_176.index.name = 'Datetime'
combined_125.index.name= 'Datetime'
soilmoisture_mbsec_hourly.index.name= 'Datetime'

In [ ]:
rmsp3_combined['10/1/24':].to_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Processed Data/rmsp3combined.csv')
combined_176['10/1/24':].to_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Processed Data/176combined.csv')
combined_125['10/1/24':].to_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Processed Data/125combined.csv')

soilmoisture_mbsec_hourly['10/1/24':].to_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Processed Data/soilmoisture_mbsec.csv')